In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import pandas as pd
import numpy as np
from utils import *

from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    roc_auc_score, 
    average_precision_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

from imblearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from imblearn.under_sampling import RandomUnderSampler



In [3]:
COLUNA_ALVO = 'Class'
RANDOM_STATE = 42

In [4]:
df = pd.read_csv('data/creditcard.csv', delimiter=',')
print("Shape:", df.shape)
print("Columns:", df.columns)

X = df.drop(COLUNA_ALVO, axis=1)
y= df[COLUNA_ALVO]

Shape: (284807, 31)
Columns: Index(['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10',
       'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20',
       'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount',
       'Class'],
      dtype='object')


In [5]:
# Check class distribution
distribution = df['Class'].value_counts()
print(distribution)

distribution = df['Class'].value_counts(normalize=True) * 100
print(distribution)

Class
0    284315
1       492
Name: count, dtype: int64
Class
0    99.827251
1     0.172749
Name: proportion, dtype: float64


In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        ('scaler', StandardScaler(), ['Time', 'Amount'])
    ],
    remainder='passthrough'
)

In [7]:
pipeline_rfc = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('sampler', RandomUnderSampler(random_state=RANDOM_STATE)),
    ('model', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
])

param_grid_rfc = {
    'model__n_estimators': [50, 100, None],
    'model__max_depth': [5, 10, None],
    'model__min_samples_leaf': [2, 5, None]
}

In [8]:
pipeline_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('sampler', RandomUnderSampler(random_state=RANDOM_STATE)),
    ('model', LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, n_jobs=-1))
])
param_grid_lr = {
    'model__C': [0.01, 0.1, 1.0],
    'model__solver': ['liblinear', 'saga']
}

In [9]:
pipeline_knn = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('sampler', RandomUnderSampler(random_state=RANDOM_STATE)),
    ('model', KNeighborsClassifier(n_jobs=-1))
])
param_grid_knn = {
    'model__n_neighbors': [3, 5, 7],
    'model__weights': ['uniform', 'distance']
}

In [10]:
models_to_run = {
    'RandomForest': {
        'estimator': pipeline_rfc,
        'param_grid': param_grid_rfc
    },
    'LogisticRegression': {
        'estimator': pipeline_lr,
        'param_grid': param_grid_lr
    },
    'KNN': {
        'estimator': pipeline_knn,
        'param_grid': param_grid_knn
    }
}

In [11]:
all_model_results = {}

for model_name, config in models_to_run.items():
    
    print(f"\n\n==============================================")
    print(f"      INICIANDO AVALIAÇÃO: {model_name}      ")
    print(f"==============================================")
    
    pipeline = config['estimator']
    param_grid = config['param_grid']
        
    inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=inner_cv,
        scoring='roc_auc',
        n_jobs=-1,
        verbose=1
    )

    # Listas para guardar resultados
    outer_roc_auc_scores = []
    outer_avg_precision_scores = []
    all_y_test = []
    all_y_pred = []

    # Loop do Ciclo Externo
    fold_num = 1
    for train_idx, test_idx in outer_cv.split(X, y):
        print(f"\n\n{model_name} - Fold Externo {fold_num}/{outer_cv.get_n_splits()}")
        
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        print(f"Rodando ciclo interon no fold {fold_num}")
        grid_search.fit(X_train, y_train)

        best_model = grid_search.best_estimator_
        print(f"Melhores parâmetros: {grid_search.best_params_}")

        y_pred = best_model.predict(X_test)
        y_pred_proba = best_model.predict_proba(X_test)[:, 1]

        roc_auc = roc_auc_score(y_test, y_pred_proba)
        avg_precision = average_precision_score(y_test, y_pred_proba)
        
        outer_roc_auc_scores.append(roc_auc)
        outer_avg_precision_scores.append(avg_precision)
        
        all_y_test.extend(y_test)
        all_y_pred.extend(y_pred)
        
        print(f"\nFold {fold_num} - ROC-AUC: {roc_auc:.4f} | Avg. Precision: {avg_precision:.4f}")
        fold_num += 1

    print(f"\n\n--- Avaliação Final para: {model_name} ---")

    mean_roc_auc = np.mean(outer_roc_auc_scores)
    std_roc_auc = np.std(outer_roc_auc_scores)
    mean_avg_precision = np.mean(outer_avg_precision_scores)
    std_avg_precision = np.std(outer_avg_precision_scores)

    print("\nPerformance Média nos 5 Folds Externos:")
    print(f"Média ROC-AUC:      {mean_roc_auc:.4f} +/- {std_roc_auc:.4f}")
    print(f"Média Avg. Precision: {mean_avg_precision:.4f} +/- {std_avg_precision:.4f}")

    print("\nMatriz de Confusão (Consolidada):")

    all_y_test_np = np.array(all_y_test)
    all_y_pred_np = np.array(all_y_pred)

    cm = confusion_matrix(all_y_test_np, all_y_pred_np)
    print(cm)
    
    print(f"Total de Fraudes Reais:    {np.sum(all_y_test_np == 1)}") 
    print(f"Total de Fraudes Pegas (VP): {cm[1, 1]}")
    print(f"Total de Fraudes Perdidas (FN): {cm[1, 0]}")

    print("\nRelatório de Classificação (Consolidado dos 5 folds):")

    print(
        classification_report(all_y_test_np, all_y_pred_np, target_names=['Classe 0 (Normal)', 'Classe 1 (Fraude)'])
    )

    report_dict = classification_report(all_y_test_np, all_y_pred_np, output_dict=True)

    # resultados médios
    all_model_results[model_name] = {
        'roc_auc': mean_roc_auc,
        'avg_precision': mean_avg_precision,
        'recall_fraude': report_dict['1']['recall'],
        'precision_fraude': report_dict['1']['precision']
    }


print("\n \n=======================================================")
print("      SUMÁRIO FINAL - COMPARAÇÃO DOS MODELOS      ")
print("=======================================================")
print("(Métricas médias calculadas a partir dos 5 folds do Nested CV)")

results_df = pd.DataFrame(all_model_results).T
results_df = results_df.sort_values(by='avg_precision', ascending=False)

print(results_df)




      INICIANDO AVALIAÇÃO: RandomForest      


RandomForest - Fold Externo 1/5
Rodando ciclo interon no fold 1
Fitting 5 folds for each of 27 candidates, totalling 135 fits
Melhores parâmetros: {'model__max_depth': None, 'model__min_samples_leaf': 2, 'model__n_estimators': 50}

Fold 1 - ROC-AUC: 0.9763 | Avg. Precision: 0.7221


RandomForest - Fold Externo 2/5
Rodando ciclo interon no fold 2
Fitting 5 folds for each of 27 candidates, totalling 135 fits
Melhores parâmetros: {'model__max_depth': None, 'model__min_samples_leaf': 2, 'model__n_estimators': 100}

Fold 2 - ROC-AUC: 0.9828 | Avg. Precision: 0.7634


RandomForest - Fold Externo 3/5
Rodando ciclo interon no fold 3
Fitting 5 folds for each of 27 candidates, totalling 135 fits
Melhores parâmetros: {'model__max_depth': None, 'model__min_samples_leaf': 2, 'model__n_estimators': 50}

Fold 3 - ROC-AUC: 0.9871 | Avg. Precision: 0.7726


RandomForest - Fold Externo 4/5
Rodando ciclo interon no fold 4
Fitting 5 folds for each of 27 ca